# Face Mask Identification - Transfer Learning & Fine-Tuning

This notebook trains a deep neural network for **Face Mask Detection** (`WithMask` vs. `WithoutMask`) using **MobileNetV2** pre-trained on ImageNet.

### Pipeline Steps:
1. **Environment Setup & Seed configuration**
2. **Dataset Loading** matching `01_data_explore.ipynb` and `02_preprocess.ipynb`

3. **Data Augmentation & MobileNetV2 Preprocessing**
4. **Phase 1: Feature Extraction** (train custom classification head with frozen backbone)
5. **Phase 2: Fine-Tuning** (unfreeze top 30 layers with low learning rate $1\times 10^{-5}$)
6. **Comprehensive Evaluation**: Confusion Matrix, Precision, Recall, F1-Score, and **mAP (Mean Average Precision / PR-AUC)** at decision threshold 0.50
7. **Precision-Recall Curve Visualization**

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Available: {gpus}")
else:
    print("Running on CPU.")

## 1. Load Dataset

In [ ]:
if os.path.exists("../data"):
    BASE_DIR = "../data"
elif os.path.exists("./data"):
    BASE_DIR = "./data"
else:
    BASE_DIR = "../data"

TRAIN_DIR = os.path.join(BASE_DIR, "Train")
VAL_DIR = os.path.join(BASE_DIR, "Validation")
TEST_DIR = os.path.join(BASE_DIR, "Test")

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    color_mode="rgb",
    shuffle=False
)

class_names = train_ds.class_names
print(f"Classes: {class_names}")

## 2. Data Augmentation and Pipeline Optimization

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
], name="data_augmentation")

AUTOTUNE = tf.data.AUTOTUNE

def prepare_train(ds):
    return (
        ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
          .cache()
          .prefetch(buffer_size=AUTOTUNE)
    )

def prepare_val_test(ds):
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds_ready = prepare_train(train_ds)
val_ds_ready = prepare_val_test(val_ds)
test_ds_ready = prepare_val_test(test_ds)

## 3. Build Transfer Learning Model with MobileNetV2

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze for Phase 1

inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="input_image")
x = preprocess_input(inputs)  # Scales pixel values to [-1, 1]
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation="sigmoid", name="predictions")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="FaceMask_MobileNetV2")
model.summary()

## 4. Phase 1: Feature Extraction

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6, verbose=1
    )
]

history_p1 = model.fit(
    train_ds_ready,
    validation_data=val_ds_ready,
    epochs=10,
    callbacks=callbacks_p1
)

## 5. Phase 2: Fine-Tuning

In [ ]:
# Unfreeze top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1
    )
]

history_finetune = model.fit(
    train_ds_ready,
    validation_data=val_ds_ready,
    epochs=10,
    callbacks=callbacks_p2
)

## 6. Evaluation on Test Set: F1, Precision, Recall & mAP50

In [ ]:
# Collect ground truth and predictions
y_true_list = []
for _, labels in test_ds:
    y_true_list.extend(labels.numpy().flatten())
y_true = np.array(y_true_list, dtype=int)

y_pred_probs = model.predict(test_ds).flatten()
y_pred = (y_pred_probs >= 0.5).astype(int)  # Decision threshold 0.50

# Metrics at threshold 0.50
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

# Classification mAP (PR-AUC across classes)
ap_class1 = average_precision_score(y_true, y_pred_probs)
ap_class0 = average_precision_score(1 - y_true, 1 - y_pred_probs)
mAP = (ap_class0 + ap_class1) / 2.0

print("=" * 50)
print(f"Weighted Precision (threshold 0.50): {precision:.4f}")
print(f"Weighted Recall    (threshold 0.50): {recall:.4f}")
print(f"Weighted F1-Score  (threshold 0.50): {f1:.4f}")
print("-" * 50)
print(f"AP for '{class_names[0]}': {ap_class0:.4f}")
print(f"AP for '{class_names[1]}': {ap_class1:.4f}")
print(f"mAP (Mean Average Precision / PR-AUC): {mAP:.4f}")
print("=" * 50)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

## 7. Precision-Recall Curve Visualization

In [ ]:
prec0, rec0, _ = precision_recall_curve(1 - y_true, 1 - y_pred_probs)
prec1, rec1, _ = precision_recall_curve(y_true, y_pred_probs)

plt.figure(figsize=(8, 6))
plt.plot(rec0, prec0, label=f"{class_names[0]} (AP = {ap_class0:.4f})", linewidth=2)
plt.plot(rec1, prec1, label=f"{class_names[1]} (AP = {ap_class1:.4f})", linewidth=2)
plt.title(f"Precision-Recall Curve (mAP = {mAP:.4f})", fontsize=14)
plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.legend(loc="lower left", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.7)
plt.show()

## 8. Save the Model

In [ ]:
os.makedirs("../models", exist_ok=True)
model_path = "../models/facemask_mobilenetv2_finetuned.keras"
model.save(model_path)
print(f"Model saved at: {model_path}")